# ISO 26262: Functional Safety for Road Vehicles

**Module 03 · Functional Safety (ISO 26262)** · Notebook 1 of 5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/03_Functional_Safety_ISO26262/01_iso26262_fundamentals.ipynb)

| Level | Time | Prerequisites | Standards |
|---|---|---|---|
| Beginner | ~45 min | None ([Module 01](../01_Perception/README.md) helpful) | ISO 26262:2018 (Parts 1–6, 9) |

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

By the end of this notebook you will be able to:

1. Describe the structure and key concepts of ISO 26262:2018
2. Outline the V-model and where each safety activity sits in it
3. Classify hazardous events by Severity, Exposure and Controllability and determine the ASIL with ISO 26262-3 Table 4
4. Explain what a HARA rates (vehicle-level malfunctioning behaviour) and what it does not (causes, safety mechanisms)
5. Separate ISO 26262 from ISO 21448 (SOTIF) and ISO/PAS 8800 for an AI-based perception function
6. Distinguish verification, safety validation and confirmation measures

In [ ]:
# Setup: only standard scientific-Python packages (all preinstalled on Colab)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

rng = np.random.default_rng(42)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
print("Setup complete.")

## 1. ISO 26262 overview

ISO 26262 is the international standard for the **functional safety** of electrical and electronic (E/E) systems in series-production road vehicles. Functional safety is the *absence of unreasonable risk due to hazards caused by malfunctioning behaviour of E/E systems* (ISO 26262-1:2018). The current edition is ISO 26262:2018 (2nd edition); a 3rd edition is in development, with publication expected around 2028 (as of October 2026, [Omnex standards update, July 2026](https://blog.omnex.com/2026/07/02/automotive-standards-update-spice-pam-4-0-iso-26262-ed-3-functional-safety-iso-21434-2/)).

### Structure of ISO 26262:2018 (12 parts)

1. **Part 1**: Vocabulary
2. **Part 2**: Management of functional safety
3. **Part 3**: Concept phase (item definition, HARA, functional safety concept)
4. **Part 4**: Product development at the system level
5. **Part 5**: Product development at the hardware level
6. **Part 6**: Product development at the software level
7. **Part 7**: Production, operation, service and decommissioning
8. **Part 8**: Supporting processes
9. **Part 9**: Automotive Safety Integrity Level (ASIL)-oriented and safety-oriented analyses
10. **Part 10**: Guidelines on ISO 26262
11. **Part 11**: Guidelines on application of ISO 26262 to semiconductors
12. **Part 12**: Adaptation of ISO 26262 for motorcycles

### Key principles

- **Risk-based approach**: the required rigour follows from a hazard analysis and risk assessment (HARA).
- **Safety lifecycle**: from concept to decommissioning.
- **ASIL grading**: four integrity levels, A (lowest) to D (highest); QM means that ordinary quality management is sufficient.
- **Two kinds of failures**: *systematic* failures (specification, design and software errors) are addressed by ASIL-dependent process and method requirements; *random hardware* failures are additionally addressed by quantitative hardware metrics (SPFM, LFM, PMHF; see [Notebook 03](03_fmea_and_fmeda.ipynb)).

## 2. The V-model

ISO 26262 organises product development along a V-model: design and specification phases on the left, verification and integration phases on the right, each linked to its counterpart by traceability. The ASIL is determined **at the very start**, in the concept phase (item definition and HARA), and then flows down into every requirement.

In [ ]:
def visualise_v_model():
    """Visualise the ISO 26262 V-model (simplified)."""
    fig, ax = plt.subplots(figsize=(13, 7))
    left_x, left_y = [1, 2, 3, 4], [5, 4, 3, 2]
    left_labels = ["Item definition\n& HARA (Part 3)", "Functional safety\nconcept (Part 3)",
                   "Technical safety concept /\nsystem architecture (Part 4)", "HW & SW design\n(Parts 5, 6)"]
    right_x, right_y = [6, 7, 8, 9], [2, 3, 4, 5]
    right_labels = ["HW & SW unit\nverification", "HW-SW & system\nintegration testing",
                    "Item integration\ntesting (vehicle)", "Safety validation\n(Part 4 cl. 8)"]
    bottom = (5, 1, "Implementation")
    ax.plot(left_x + [bottom[0]], left_y + [bottom[1]], "o-", lw=3, ms=12, color="#0072B2", label="Specification and design")
    ax.plot([bottom[0]] + right_x, [bottom[1]] + right_y, "o-", lw=3, ms=12, color="#D55E00", label="Verification and validation")
    # Traceability: each design level is verified at the same height on the right
    for i in range(4):
        ax.plot([left_x[i], right_x[3 - i]], [left_y[i], right_y[3 - i]], "--", lw=1.2, color="grey", alpha=0.6)
    for x, y, lab in zip(left_x, left_y, left_labels):
        ax.text(x, y + 0.25, lab, ha="center", va="bottom", fontsize=9, bbox=dict(boxstyle="round", facecolor="#cfe5f3"))
    for x, y, lab in zip(right_x, right_y, right_labels):
        ax.text(x, y + 0.25, lab, ha="center", va="bottom", fontsize=9, bbox=dict(boxstyle="round", facecolor="#f6d8c4"))
    ax.text(bottom[0], bottom[1] - 0.45, bottom[2], ha="center", va="top", fontsize=9,
            bbox=dict(boxstyle="round", facecolor="#e0e0e0"))
    ax.annotate("ASIL determined here\n(HARA, ISO 26262-3 cl. 6)", xy=(1, 5), xytext=(0.0, 6.3),
                fontsize=10, arrowprops=dict(arrowstyle="->"), bbox=dict(boxstyle="round", facecolor="#F0E442"))
    ax.text(9.6, 6.4, "Confirmation measures (Part 2):\nreviews, audit, assessment", ha="center", fontsize=9,
            bbox=dict(boxstyle="round", facecolor="#ffffff", edgecolor="grey"))
    ax.set_xlim(-0.5, 11); ax.set_ylim(0, 7)
    ax.axis("off")
    ax.set_title("ISO 26262 V-model (simplified)", fontsize=14, fontweight="bold")
    ax.legend(loc="lower right")
    plt.tight_layout(); plt.show()
    return left_labels + [bottom[2]], [bottom[2]] + right_labels

left, right = visualise_v_model()

# Accessibility summary
print("--- V-model (figure summary) ---")
print("Left branch, top to bottom:", " -> ".join(l.replace("\n", " ") for l in left))
print("Right branch, bottom to top:", " -> ".join(r.replace("\n", " ") for r in right))
print("Dashed lines link each design level to the verification activity at the same level;")
print("the ASIL is determined at the top left, in the HARA.")

## 3. ASIL classification

The **Automotive Safety Integrity Level (ASIL)** of a hazardous event is determined from three parameters (ISO 26262-3:2018, clause 6 and Annex B). The descriptions below follow the wording of the standard.

**Severity (S)**: the extent of harm to each person potentially affected (Annex B gives an informative mapping to the Abbreviated Injury Scale, AIS).

| Class | Description | Informative AIS guidance |
|---|---|---|
| S0 | No injuries | AIS 0, or less than 10 % probability of AIS 1–6 |
| S1 | Light and moderate injuries | More than 10 % probability of AIS 1–6 (and not S2 or S3) |
| S2 | Severe and life-threatening injuries (survival probable) | More than 10 % probability of AIS 3–6 (and not S3) |
| S3 | Life-threatening injuries (survival uncertain), fatal injuries | More than 10 % probability of AIS 5–6 |

**Exposure (E)**: the probability of being in the operational situation in which the hazard can lead to harm. It is rated **either by duration** (share of the average operating time) **or by frequency** of the situation, not per hour of failure.

| Class | Description | Duration (share of average operating time) | Frequency |
|---|---|---|---|
| E0 | Incredible | – | – (no ASIL assigned) |
| E1 | Very low probability | Not specified | Less often than once a year for the great majority of drivers |
| E2 | Low probability | < 1 % | A few times a year for the great majority of drivers |
| E3 | Medium probability | 1 % to 10 % | Once a month or more often for an average driver |
| E4 | High probability | > 10 % | All or almost every drive on average |

**Controllability (C)**: the ability of the **driver or other persons at risk** (e.g. other road users) to avoid the specific harm through timely reactions.

| Class | Description | Guidance |
|---|---|---|
| C0 | Controllable in general | No percentage; the situation is generally controllable (no ASIL assigned) |
| C1 | Simply controllable | 99 % or more of all drivers or other traffic participants are usually able to avoid harm |
| C2 | Normally controllable | 90 % or more are usually able to avoid harm |
| C3 | Difficult to control or uncontrollable | Less than 90 % are usually able, or barely able, to avoid harm |

**ASIL levels**: QM (no ASIL; quality management suffices), then ASIL A (least stringent) to ASIL D (most stringent).

In [ ]:
# ISO 26262-3:2018 Table 4 -- ASIL determination. ASIL_TABLE[S][E] -> (C1, C2, C3)
# Identical to avsafety.iso26262.ASIL_TABLE (see the repository's avsafety/ package).
ASIL_ORDER = ("QM", "A", "B", "C", "D")
ASIL_TABLE = {
    "S1": {"E1": ("QM", "QM", "QM"), "E2": ("QM", "QM", "QM"), "E3": ("QM", "QM", "A"), "E4": ("QM", "A", "B")},
    "S2": {"E1": ("QM", "QM", "QM"), "E2": ("QM", "QM", "A"), "E3": ("QM", "A", "B"), "E4": ("A", "B", "C")},
    "S3": {"E1": ("QM", "QM", "A"), "E2": ("QM", "A", "B"), "E3": ("A", "B", "C"), "E4": ("B", "C", "D")},
}
_VALID = {"S": ("S0", "S1", "S2", "S3"), "E": ("E0", "E1", "E2", "E3", "E4"), "C": ("C0", "C1", "C2", "C3")}


def determine_asil(severity: str, exposure: str, controllability: str) -> str:
    """Return the ASIL ("QM", "A".."D") of a hazardous event (ISO 26262-3:2018 Table 4).

    S0, E0 or C0 mean that no ASIL is assigned (returned as "QM").
    Invalid classes raise ValueError -- a typo must never silently become "QM".
    The same function is available as avsafety.iso26262.determine_asil.
    """
    s, e, c = (str(x).strip().upper() for x in (severity, exposure, controllability))
    for val, key in ((s, "S"), (e, "E"), (c, "C")):
        if val not in _VALID[key]:
            raise ValueError(f"invalid {key} class: {val!r}; expected one of {_VALID[key]}")
    if s == "S0" or e == "E0" or c == "C0":
        return "QM"
    return ASIL_TABLE[s][e][int(c[1]) - 1]


def asil_rank(asil: str) -> int:
    """Numeric rank of an ASIL (QM=0 ... D=4)."""
    return ASIL_ORDER.index(asil)


def visualise_asil_matrix():
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
    titles = {"S1": "S1: light and moderate injuries", "S2": "S2: severe injuries (survival probable)",
              "S3": "S3: life-threatening or fatal"}
    for ax, sev in zip(axes, ["S1", "S2", "S3"]):
        m = np.array([[asil_rank(a) for a in ASIL_TABLE[sev][e]] for e in ["E1", "E2", "E3", "E4"]])
        ax.imshow(m, cmap="viridis", vmin=0, vmax=4, aspect="auto")
        for i, e in enumerate(["E1", "E2", "E3", "E4"]):
            for j, a in enumerate(ASIL_TABLE[sev][e]):
                ax.text(j, i, a, ha="center", va="center", fontsize=14, fontweight="bold",
                        color="white" if asil_rank(a) < 3 else "black")
        ax.set_xticks(range(3)); ax.set_xticklabels(["C1", "C2", "C3"])
        ax.set_yticks(range(4)); ax.set_yticklabels(["E1", "E2", "E3", "E4"])
        ax.set_xlabel("Controllability class"); ax.set_ylabel("Exposure class")
        ax.set_title(titles[sev], fontsize=11); ax.grid(False)
    plt.suptitle("ASIL determination, ISO 26262-3:2018 Table 4", fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()

visualise_asil_matrix()

# Accessibility summary, generated from the table itself
print("--- ASIL determination table (text summary) ---")
for sev in ["S1", "S2", "S3"]:
    print(f"  {sev}: " + "; ".join(f"{e}: " + "/".join(ASIL_TABLE[sev][e]) for e in ["E1", "E2", "E3", "E4"])
          + "   (values for C1/C2/C3)")
print("  S0, E0 or C0 -> no ASIL (QM).")

# A handy check: the table follows the 'sum rule' S+E+C = 10 -> D, 9 -> C, 8 -> B, 7 -> A, otherwise QM
ok = all(determine_asil(f"S{s}", f"E{e}", f"C{c}") == {10: "D", 9: "C", 8: "B", 7: "A"}.get(s + e + c, "QM")
         for s in (1, 2, 3) for e in (1, 2, 3, 4) for c in (1, 2, 3))
print(f"\nSum rule reproduces Table 4 for all 36 cells: {ok}")

# Invalid input must raise instead of silently returning QM
for bad in [("S3", "E5", "C3"), ("S3", "E4", "X")]:
    try:
        determine_asil(*bad)
    except ValueError as err:
        print(f"determine_asil{bad} -> ValueError: {err}")

**Worked examples.** The hazardous events below are written as *vehicle-level* behaviour in a stated situation, not as component failures.

In [ ]:
examples = [
    ("S3", "E4", "C3", "Unintended full AEB braking on a motorway at 130 km/h with a closely following vehicle"),
    ("S3", "E4", "C2", "Same, but rated C2 (following drivers assumed normally able to react)"),
    ("S2", "E3", "C2", "Unintended lane-keeping steering torque on a rural road"),
    ("S1", "E2", "C1", "Unintended short warning-chime activation of a parking sensor"),
]
for s, e, c, situation in examples:
    a = determine_asil(s, e, c)
    print(f"{situation}\n   {s}, {e}, {c} -> {'QM (no ASIL)' if a == 'QM' else 'ASIL ' + a}")
print("\nNote how one controllability class changes the result by one ASIL:"
      " ratings need a written rationale (see Notebook 02).")

> **Where ISO 26262 ends: the ISO 26262 / ISO 21448 / ISO/PAS 8800 boundary**
>
> | Question | Standard | Pedestrian-AEB example |
> |---|---|---|
> | Does the vehicle behave hazardously **because an E/E element is faulty** (random hardware fault or systematic hardware/software fault)? | **ISO 26262:2018** (functional safety) | A bit flip in the camera ECU's RAM corrupts the object list; a broken brake-request line; a software bug in the time-to-collision computation. AEB does not brake for a pedestrian, or brakes for nothing. |
> | Does the vehicle behave hazardously **although nothing is broken**, because the intended function or its specification is insufficient in some situation (a *triggering condition*)? | **ISO 21448:2022** (SOTIF) | The fault-free camera is dazzled by low sun, or the pedestrian is partly occluded by a parked van, and is not detected in time. |
> | Is the insufficiency **specific to a machine-learning component** (training data, model generalisation, robustness, uncertainty)? | **ISO/PAS 8800:2024** (safety and AI), which refines ISO 26262 and ISO 21448 for AI elements | The detector was trained with too few children or wheelchair users and misses them; its confidence is badly calibrated in rain. |
>
> The *same* vehicle-level hazard ("AEB does not brake when a collision with a pedestrian is imminent") can therefore have causes in all three standards. ISO 26262 rates the hazard in the HARA without asking for the cause; the **causes** are then treated by the standard that owns them. A HARA does not turn a perception performance limit into an ISO 26262 fault, and ASIL-graded process rigour alone cannot achieve a detection-rate target. For a review of how the SOTIF literature handles this boundary, see Patel, Jung and Khatun (2025), *A Systematic Literature Review on Safety of the Intended Functionality for Automated Driving Systems*, SAE Technical Paper 2025-01-5030, https://doi.org/10.4271/2025-01-5030.

## 4. Hazard analysis and risk assessment (HARA)

The HARA (ISO 26262-3:2018, clause 6) identifies and classifies the hazardous events of an **item** and formulates **safety goals**.

### HARA process steps

1. **Item definition** (clause 5): the function at vehicle level, its boundary, interfaces and assumptions.
2. **Situation analysis**: operational situations and operating modes in which a malfunction could lead to harm.
3. **Hazard identification**: vehicle-level *malfunctioning behaviours* (e.g. "AEB does not brake when required", "AEB brakes unintentionally") and the resulting hazardous events.
4. **Classification**: S, E and C for each hazardous event, with rationale.
5. **ASIL determination**: with Table 4.
6. **Safety goals**: one top-level safety requirement per hazardous event (or group of similar events), with the highest ASIL of the events it covers, a safe state where applicable, and the FTTI.

Three rules an assessor will check:

- The HARA is done on the **item at vehicle level**; hazards are vehicle behaviours, not sensor outputs.
- It is done **without considering the safety mechanisms** of the item that is being analysed.
- Each hazardous event is rated for the **worst credible** operational situation, with a documented rationale.

The example below is a short version of the HARA worked through in detail in [Notebook 02](02_hara.ipynb).

> **Illustrative data:** the ratings below are teaching values with plausible rationale, not the result of an actual HARA for a real product.

In [ ]:
class HARAAnalysis:
    """Minimal HARA record: hazardous event = malfunctioning behaviour x operational situation."""

    def __init__(self, item_name: str):
        self.item_name = item_name
        self.events = []

    def add_hazardous_event(self, he_id, malfunctioning_behaviour, situation, severity, exposure,
                            controllability, safety_goal_id):
        self.events.append({
            "id": he_id, "malfunctioning_behaviour": malfunctioning_behaviour, "situation": situation,
            "S": severity, "E": exposure, "C": controllability,
            "ASIL": determine_asil(severity, exposure, controllability), "SG": safety_goal_id})

    def to_dataframe(self):
        return pd.DataFrame(self.events)


hara = HARAAnalysis("AEB function (perception + decision + brake request)")
hara.add_hazardous_event("HE-01", "No AEB braking when a collision is imminent (omission)",
                         "Urban, 30-50 km/h, pedestrian steps into path", "S3", "E2", "C2", "SG-02")
hara.add_hazardous_event("HE-02", "No AEB braking when a collision is imminent (omission)",
                         "Motorway, 80-130 km/h, approaching stationary end of a traffic jam", "S3", "E3", "C2", "SG-02")
hara.add_hazardous_event("HE-03", "Unintended full AEB braking (commission)",
                         "Motorway, 100-130 km/h, following vehicle at short headway", "S3", "E4", "C3", "SG-01")
hara.add_hazardous_event("HE-04", "Unintended full AEB braking (commission)",
                         "Urban, 30-50 km/h, following vehicle close behind", "S1", "E4", "C2", "SG-01")
df_hara = hara.to_dataframe()
display(df_hara)

# Safety goals: vehicle-level goals, ASIL = highest ASIL of the events they cover
sg_text = {
    "SG-01": ("Avoid unintended AEB deceleration",
              "AEB brake request inhibited; AEB-unavailable warning; driver keeps full control"),
    "SG-02": ("Avoid omission of AEB deceleration when a collision with a pedestrian or a stationary vehicle is imminent",
              "AEB unavailable with driver warning; driver remains responsible for braking"),
}
goals = (df_hara.groupby("SG")["ASIL"].agg(lambda a: max(a, key=asil_rank)).reset_index())
goals["Safety goal"] = goals["SG"].map(lambda g: sg_text[g][0])
goals["Safe state"] = goals["SG"].map(lambda g: sg_text[g][1])
goals["Covers"] = goals["SG"].map(lambda g: ", ".join(df_hara.loc[df_hara.SG == g, "id"]))
display(goals[["SG", "Safety goal", "ASIL", "Safe state", "Covers"]])

# Accessibility summary
print("--- HARA (text summary) ---")
for _, r in df_hara.iterrows():
    print(f"  {r.id}: {r.malfunctioning_behaviour} | {r.situation} | {r.S}/{r.E}/{r.C} -> ASIL {r.ASIL}")
for _, g in goals.iterrows():
    print(f"  {g.SG} [ASIL {g.ASIL}] {g['Safety goal']} (covers {g.Covers}); safe state: {g['Safe state']}")
omission = df_hara[df_hara.malfunctioning_behaviour.str.contains("omission")]["ASIL"].map(asil_rank).max()
commission = df_hara[df_hara.malfunctioning_behaviour.str.contains("commission")]["ASIL"].map(asil_rank).max()
dominant = "commission (unintended braking)" if commission > omission else "omission (no braking)"
print(f"\nHighest ASIL for omission: {ASIL_ORDER[omission]}; for commission: {ASIL_ORDER[commission]}."
      f" With these ratings the {dominant} hazard drives the ASIL. For a driver-assistance AEB this is typical:"
      " when AEB merely fails to act, the driver is still responsible for braking.")

## 5. From safety goals to functional safety requirements

Each safety goal is refined into **functional safety requirements (FSR)** in the functional safety concept (ISO 26262-3, clause 7), and then into **technical safety requirements (TSR)** in the technical safety concept (ISO 26262-4). Requirements inherit the ASIL of the safety goal unless **ASIL decomposition** (ISO 26262-9, clause 5) is applied to sufficiently independent, redundant elements.

**Example for SG-01 "Avoid unintended AEB deceleration" [ASIL D]:**

- **FSR-01.1**: A brake request from the AEB decision function shall be executed only if an independent plausibility channel confirms a collision threat. [ASIL D, decomposed into B(D) for the decision channel and B(D) for the plausibility channel; see [Notebook 02](02_hara.ipynb), section 10]
- **FSR-01.2**: A loss or corruption of the perception output (e.g. stale or inconsistent object lists) shall be detected within the fault detection time interval, and the AEB brake request shall then be inhibited and the driver warned (safe state). [ASIL D]
- **FSR-01.3**: The brake request signal transmitted to the brake system shall be protected against corruption, repetition and loss (end-to-end protection). [ASIL D]

**Example for SG-02 "Avoid omission of AEB deceleration" [ASIL B]:**

- **FSR-02.1**: Faults of the sensors, the AEB electronic control unit (ECU) or the brake-request path that prevent AEB braking shall be detected and indicated to the driver within the fault-handling time. [ASIL B]

> **Performance targets are not ISO 26262 requirements.** Statements such as "detect pedestrians within 50 m with 99.9 % recall" or "the CNN shall achieve a 99.9 % true-positive rate" describe the *nominal performance* of the intended function. Their acceptance criteria and validation targets belong to **ISO 21448** (SOTIF) and, for the ML component, **ISO/PAS 8800**. ASIL-graded process rigour cannot achieve them. Likewise, a camera resolution of 1920 × 1080 at 30 fps is a design parameter, not a safety requirement.

## 6. AI/ML components: which standard covers what

| Topic | Main home | Comment |
|---|---|---|
| Training-data coverage of the operational design domain (ODD), labelling quality, dataset versioning | ISO/PAS 8800:2024 | Data are a new kind of work product without an ISO 26262 equivalent |
| Model robustness, generalisation, corner cases, adversarial examples | ISO 21448:2022 / ISO/PAS 8800:2024 | Functional insufficiencies and triggering conditions |
| Confidence scoring, out-of-distribution (OOD) detection as runtime monitors | ISO 21448 / ISO/PAS 8800 | Measures against performance insufficiencies |
| Hardware faults in the accelerator, memory corruption of weights, timing overruns | ISO 26262:2018 | Classic E/E faults: diagnostics, hardware metrics, watchdogs |
| Software development of the code that runs the model | ISO 26262-6:2018 | ISO 26262-6 assumes specifiable, analysable code; ISO/PAS 8800 adds AI-specific guidance |
| Graceful degradation, driver takeover, minimal-risk manoeuvre | ISO 26262 (safe states) and ISO 21448 (system behaviour) | Behaviour must be specified in both analyses |
| Updates and retraining (OTA) | All three, plus UN R156 | Impact analysis; the safety case must be re-established |

The dashboard below separates three runtime signals by the standard that owns them.

> **Illustrative data:** values below are synthetic, chosen for teaching; they are not measurements.

In [ ]:
def simulate_runtime_monitoring(n=100, event=(60, 70)):
    """Synthetic runtime signals of a camera-based perception channel."""
    a, b = event
    confidence = np.clip(0.95 + 0.03 * rng.standard_normal(n), 0.7, 1.0)
    confidence[a:b] = 0.78 + 0.03 * rng.standard_normal(b - a)      # e.g. low sun: SOTIF triggering condition
    latency = np.clip(50 + 8 * rng.standard_normal(n), 30, 120)
    latency[a:b] = 85 + 10 * rng.standard_normal(b - a)             # e.g. thermal throttling: E/E-related
    health = np.ones(n); health[a:b] = 0.8                         # e.g. image-sensor diagnostic flag
    return confidence, latency, health

confidence, latency, health = simulate_runtime_monitoring()
t = np.arange(len(confidence))
LIMITS = {"confidence": 0.85, "latency_ms": 90.0, "health": 0.9}

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
axes[0].plot(t, confidence, color="#0072B2", lw=2, label="Detection confidence")
axes[0].axhline(LIMITS["confidence"], color="#D55E00", ls="--", label="Monitor threshold")
axes[0].set_ylabel("Confidence (0-1)")
axes[0].set_title("ML confidence monitor: measure against performance insufficiency (ISO 21448 / ISO/PAS 8800)")
axes[1].plot(t, latency, color="#009E73", lw=2, label="Processing latency")
axes[1].axhline(LIMITS["latency_ms"], color="#D55E00", ls="--", label="Deadline monitor")
axes[1].set_ylabel("Latency (ms)")
axes[1].set_title("Timing monitor (watchdog): E/E diagnostic (ISO 26262)")
axes[2].plot(t, health, color="#CC79A7", lw=2, label="Sensor health index")
axes[2].axhline(LIMITS["health"], color="#D55E00", ls="--", label="Diagnostic threshold")
axes[2].set_ylabel("Health index (0-1)"); axes[2].set_xlabel("Time (frames)")
axes[2].set_title("Sensor self-diagnostic: E/E diagnostic (ISO 26262)")
for ax in axes:
    ax.legend(loc="lower left", fontsize=8)
plt.tight_layout(); plt.show()

# Text summary and conclusion derived from the data
viol = {
    "confidence below threshold": np.flatnonzero(confidence < LIMITS["confidence"]),
    "latency above deadline": np.flatnonzero(latency > LIMITS["latency_ms"]),
    "sensor health below threshold": np.flatnonzero(health < LIMITS["health"]),
}
print("--- Runtime monitoring (text summary) ---")
print(f"Mean confidence {confidence.mean():.3f}, min {confidence.min():.3f} at frame {confidence.argmin()}")
print(f"Mean latency {latency.mean():.1f} ms, max {latency.max():.1f} ms at frame {latency.argmax()}")
for name, idx in viol.items():
    span = f"frames {idx.min()}-{idx.max()}" if idx.size else "none"
    print(f"  {name}: {idx.size} frames ({span})")
if viol["latency above deadline"].size or viol["sensor health below threshold"].size:
    print("Reaction: an E/E diagnostic fired -> inhibit the AEB brake request and warn the driver (safe state of SG-01/SG-02).")
elif viol["confidence below threshold"].size:
    print("Reaction: only the ML confidence monitor fired -> degrade per the SOTIF concept (e.g. warn, limit function).")
else:
    print("No monitor fired.")

## 7. Verification, safety validation and confirmation measures

ISO 26262 uses three different kinds of evidence; they are often confused.

| Activity | Question | Where in ISO 26262 | Examples |
|---|---|---|---|
| **Verification** | Does this work product fulfil its specified requirements? | Throughout Parts 3–6, 8 (clause 9) | Reviews, analyses, simulation, unit/integration tests, fault injection |
| **Safety validation** | Are the safety goals achieved at vehicle level, and are the functional and technical safety concepts appropriate? | Part 4, clause 8 | Vehicle tests, long-term tests, user tests, fault injection at vehicle level |
| **Confirmation measures** | Are the work products, the process and the achieved safety adequate? | Part 2, clause 6 (Table 1) | Confirmation reviews, functional-safety audit, functional-safety assessment |

Confirmation measures are carried out with a required **level of independence** (I0 to I3) that increases with the ASIL. They are not limited to ASIL D.

### Methods depend on the ASIL

ISO 26262 lists methods in tables and marks each one per ASIL as **++** (highly recommended), **+** (recommended) or **o** (no recommendation for or against its use). There is no QM column: QM elements are handled by the quality-management system. A subset of the dynamic test methods for **software unit verification**, ISO 26262-6:2018 Table 7:

| Method (ISO 26262-6:2018 Table 7, excerpt) | ASIL A | ASIL B | ASIL C | ASIL D |
|---|---|---|---|---|
| Requirements-based test | ++ | ++ | ++ | ++ |
| Interface test | ++ | ++ | ++ | ++ |
| Fault injection test | + | + | + | ++ |
| Resource usage evaluation | + | + | + | ++ |
| Back-to-back comparison test between model and code (if applicable) | + | + | ++ | ++ |

Legend: ++ highly recommended, + recommended, o no recommendation for or against.
The table values were transcribed from secondary sources; check them against your licensed copy of ISO 26262-6:2018. Table 7 also lists review and static-analysis methods, not shown here. [Notebook 05](05_verification_and_validation.ipynb) covers the structural-coverage table (Table 9).

In [ ]:
# ISO 26262-6:2018 Table 9: structural coverage metrics at the software unit level.
# Recommendation levels only -- the standard sets NO fixed coverage percentage.
COVERAGE_TABLE_9 = pd.DataFrame(
    {"A": ["++", "+", "+"], "B": ["++", "++", "+"], "C": ["+", "++", "+"], "D": ["+", "++", "++"]},
    index=["Statement coverage", "Branch coverage", "MC/DC"])
LEVEL = {"o": 0, "+": 1, "++": 2}

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.imshow(np.vectorize(LEVEL.get)(COVERAGE_TABLE_9.values), cmap="cividis", vmin=0, vmax=2, aspect="auto")
for i in range(3):
    for j in range(4):
        v = COVERAGE_TABLE_9.iat[i, j]
        ax.text(j, i, v, ha="center", va="center", fontsize=14, fontweight="bold",
                color="black" if v == "++" else "white")
ax.set_xticks(range(4)); ax.set_xticklabels([f"ASIL {a}" for a in COVERAGE_TABLE_9.columns])
ax.set_yticks(range(3)); ax.set_yticklabels(COVERAGE_TABLE_9.index); ax.grid(False)
ax.set_title("ISO 26262-6:2018 Table 9 (++ highly recommended, + recommended)")
plt.tight_layout(); plt.show()

print("--- Structural coverage recommendations (text summary) ---")
for metric, row in COVERAGE_TABLE_9.iterrows():
    print(f"  {metric}: " + ", ".join(f"ASIL {a} {v}" for a, v in row.items()))
hr_d_only = [m for m, r in COVERAGE_TABLE_9.iterrows() if r["D"] == "++" and r["C"] != "++"]
print(f"Highly recommended only at ASIL D: {hr_d_only}")
print("Coverage targets (e.g. 100 % branch) are project decisions; gaps must be analysed and justified.")

## 8. HARA worksheet (pre-filled scenarios)

The worksheet below rates four pre-filled hazardous events from other driver-assistance functions. Change the S, E and C classes (or add rows) and re-run the cell to see how sensitive the ASIL is to each parameter.

> **Illustrative data:** the ratings are teaching values, not results of a real HARA.

In [ ]:
scenarios = [
    {"Item": "Lane keeping assist", "Hazardous event": "Unintended steering torque on a motorway, 120 km/h",
     "S": "S3", "E": "E4", "C": "C2"},
    {"Item": "Adaptive cruise control", "Hazardous event": "Unintended strong deceleration, rural road, following traffic",
     "S": "S2", "E": "E3", "C": "C2"},
    {"Item": "Adaptive cruise control", "Hazardous event": "Unintended acceleration towards a slower lead vehicle",
     "S": "S3", "E": "E3", "C": "C2"},
    {"Item": "Blind-spot warning", "Hazardous event": "No warning during a lane change with a vehicle in the blind spot",
     "S": "S2", "E": "E3", "C": "C1"},
]
df_ws = pd.DataFrame(scenarios)
df_ws["ASIL"] = [determine_asil(r.S, r.E, r.C) for r in df_ws.itertuples()]
display(df_ws)

fig, ax = plt.subplots(figsize=(10, 4.5))
ranks = df_ws["ASIL"].map(asil_rank)
bars = ax.bar(range(len(df_ws)), ranks, color=plt.cm.viridis(ranks / 4), edgecolor="black")
for b, a in zip(bars, df_ws["ASIL"]):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.05, a if a == "QM" else f"ASIL {a}",
            ha="center", va="bottom", fontweight="bold")
ax.set_xticks(range(len(df_ws)))
ax.set_xticklabels([f"{r.Item}\n({r.S}/{r.E}/{r.C})" for r in df_ws.itertuples()], fontsize=9)
ax.set_yticks(range(5)); ax.set_yticklabels(ASIL_ORDER); ax.set_ylim(0, 4.6)
ax.set_ylabel("ASIL"); ax.set_title("HARA worksheet: ASIL per hazardous event")
plt.tight_layout(); plt.show()

print("--- HARA worksheet (text summary) ---")
for _, r in df_ws.iterrows():
    print(f"  {r['Item']}: {r['Hazardous event']} | {r['S']}/{r['E']}/{r['C']} -> {r['ASIL']}")

## Key takeaways

- **Scope:** ISO 26262 addresses hazards caused by *malfunctioning behaviour of E/E systems* (random hardware and systematic faults). Performance insufficiencies of the intended function belong to ISO 21448; ML-specific insufficiencies to ISO/PAS 8800.
- **ASIL** follows from S, E and C via ISO 26262-3 Table 4. The table is one-dimensional in disguise: S + E + C = 10 → D, 9 → C, 8 → B, 7 → A, otherwise QM.
- **HARA** rates vehicle-level malfunctioning behaviour in operational situations, without crediting the item's own safety mechanisms, and needs a written rationale for every rating.
- **Safety goals** are vehicle-level goals with a concrete safe state and an FTTI. Perception performance targets are not safety goals.
- **Verification ≠ validation ≠ confirmation:** requirements fulfilled, safety goals achieved at vehicle level, adequacy confirmed independently.
- **Method tables** use ++ / + / o per ASIL. They prescribe no fixed coverage percentages.

## Self-assessment

1. Which three parameters determine the ASIL, and what does each measure?
2. Why is the ASIL determined at the top left of the V-model, and what happens to it further down?
3. Unintended full AEB braking on a motorway is rated S3/E4/C3. What ASIL results? What would C2 give?
4. A colleague writes the safety goal "The camera shall detect all pedestrians with 99.9 % recall [ASIL D]". Name two problems.
5. A pedestrian is missed because of sun glare, with all hardware and software fault-free. Which standard covers this, and why does the HARA still matter?
6. What is the difference between verification, safety validation and a functional-safety assessment?

<details>
<summary>Answers</summary>

1. Severity (extent of harm), Exposure (probability of the operational situation, rated by duration or frequency) and Controllability (ability of the driver or other persons at risk to avoid the harm).
2. The ASIL is a property of the hazardous events and safety goals found in the HARA (concept phase). It is inherited by the functional and technical safety requirements and determines which methods and how much rigour apply at every lower level, unless ASIL decomposition is applied.
3. ASIL D; with C2 the result is ASIL C.
4. (a) It is a component performance requirement, not a vehicle-level safety goal. (b) A recall target is a SOTIF/ISO/PAS 8800 acceptance criterion and cannot be achieved by ASIL-graded process measures. (c) "All pedestrians" is not verifiable. A safety goal would be "Avoid omission of AEB deceleration when a collision with a pedestrian is imminent [ASIL x]".
5. ISO 21448 (SOTIF): a functional insufficiency under a triggering condition, with no E/E fault. The HARA still identifies the vehicle-level hazard and its severity, which ISO 21448 reuses to judge whether the residual risk is acceptable.
6. Verification shows that a work product fulfils its requirements. Safety validation (Part 4, clause 8) shows at vehicle level that the safety goals are achieved and the concepts are appropriate. The functional-safety assessment (Part 2) is a confirmation measure, performed with ASIL-dependent independence, that judges whether the achieved functional safety is adequate.
</details>

## References

- ISO 26262-1 to -12:2018, *Road vehicles: Functional safety*. Geneva: ISO.
- ISO 21448:2022, *Road vehicles: Safety of the intended functionality*. Geneva: ISO.
- ISO/PAS 8800:2024, *Road vehicles: Safety and artificial intelligence*. Geneva: ISO. https://www.iso.org/standard/83303.html
- SAE J3016 (2021), *Taxonomy and Definitions for Terms Related to Driving Automation Systems for On-Road Motor Vehicles*.
- SAE J2980, *Considerations for ISO 26262 ASIL Hazard Classification*. SAE International.
- Salay, R., Queiroz, R., Czarnecki, K. (2017). *An Analysis of ISO 26262: Using Machine Learning Safely in Automotive Software*. arXiv:1709.02435. https://arxiv.org/abs/1709.02435
- Patel, M., Jung, R., Khatun, M. (2025). *A Systematic Literature Review on Safety of the Intended Functionality for Automated Driving Systems*. SAE Technical Paper 2025-01-5030. https://doi.org/10.4271/2025-01-5030

---
**Navigation:** ← [Module 03 overview](README.md) · [Module index](README.md) · [02 Hazard analysis and risk assessment](02_hara.ipynb) →